# Bước 1. Lấy những feature cơ bản. (module 1. ibtracs_preprocessing)

In [3]:
import numpy as np
import pandas as pd
import os

os.chdir(r"c:\Users\Huy\Documents\UIT\Sem_4\DS102")

In [4]:
folder = 'data/raw'
PATH = os.path.join(folder, 'ibtracs.csv')
df = pd.read_csv(
    PATH,
    engine="python",
    on_bad_lines="skip",
)
df = df.iloc[1:]

In [5]:
features = [
    "SID",              # ID của bão
    "SEASON",           # Mùa
    "NAME",             # Tên bão
    "ISO_TIME",         # Thời gian (định dạng ISO)
    "LAT",              # Vĩ độ
    "LON",              # Kinh độ
    'BASIN',            # Vùng bão
    'SUBBASIN',         # Vùng phụ bão
    "WMO_WIND",         # Tốc độ gió (theo WMO)
    "WMO_PRES",         # Áp suất (theo WMO)
    "DIST2LAND",        # Khoảng cách đến đất liền
    "STORM_SPEED",      # Tốc độ di chuyển của bão
    "STORM_DIR",        # Hướng di chuyển của bão (theo độ, 0-360)
]
df = df[features]
df.head()

,SID,SEASON,NAME,ISO_TIME,LAT,LON,BASIN,SUBBASIN,WMO_WIND,WMO_PRES,DIST2LAND,STORM_SPEED,STORM_DIR
1,1842298N11080,1842,UNNAMED,1842-10-25 03:00:00,10.9,80.3,NI,BB,,,43,9,265
2,1842298N11080,1842,UNNAMED,1842-10-25 06:00:00,10.9,79.8,NI,BB,,,0,9,265
3,1842298N11080,1842,UNNAMED,1842-10-25 09:00:00,10.8,79.4,NI,BB,,,0,9,265
4,1842298N11080,1842,UNNAMED,1842-10-25 12:00:00,10.8,78.9,NI,BB,,,0,9,265
5,1842298N11080,1842,UNNAMED,1842-10-25 15:00:00,10.8,78.4,NI,BB,,,0,9,270


In [6]:
to_num_cols = ['WMO_WIND','WMO_PRES','DIST2LAND',
               'STORM_SPEED','STORM_DIR', 'SEASON', 'LAT', 'LON']
for col in to_num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['ISO_TIME'] = pd.to_datetime(df['ISO_TIME'], errors='coerce')
df['BASIN']    = df['BASIN'].astype(str)
df['SUBBASIN'] = df['SUBBASIN'].astype(str)
df['NAME']     = df['NAME'].astype(str)

In [7]:
df = df[(df['SEASON'] <= 2025) & 
        (df['SEASON'] >= 2000) & 
        (df['BASIN'] == 'WP')]

df = df[df['ISO_TIME'].dt.hour % 6 == 0]
df = df[df['WMO_WIND'] >= 30]

print(f"Rows after filter: {len(df):,}")
print(f"Unique storms: {df['SID'].nunique()}")

Rows after filter: 12,311
Unique storms: 603


In [8]:
storm_time = df.groupby('SID')['ISO_TIME'].agg(['min','max'])
storm_time['duration'] = storm_time['max'] - storm_time['min']

more3day = storm_time[storm_time['duration'] > pd.Timedelta(days=3)].index.tolist()
df = df[df['SID'].isin(more3day)]

print(f"Số bão cuối cùng (WP, >3 ngày): {df['SID'].nunique()}")

Số bão cuối cùng (WP, >3 ngày): 418


In [9]:
df['YEAR'] = df['SEASON'].astype(int)
df['MONTH'] = df['ISO_TIME'].dt.month
df['DAY'] = df['ISO_TIME'].dt.day
df['HOUR'] = df['ISO_TIME'].dt.hour

In [10]:
drop_cols = ['BASIN', 'SUBBASIN', 'ISO_TIME', 'NAME', 'SEASON', 'STORM_SPEED', 'STORM_DIR']
df = df.drop(columns=drop_cols)

cols = ['SID', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'LAT', 'LON',
        'WMO_WIND', 'WMO_PRES', 'DIST2LAND']
df = df[cols]
df.head()

,SID,YEAR,MONTH,DAY,HOUR,LAT,LON,WMO_WIND,WMO_PRES,DIST2LAND
563182,2000125N06136,2000,5,7,0,13.1,131.7,45.0,992.0,680
563184,2000125N06136,2000,5,7,6,13.4,131.4,55.0,985.0,654
563186,2000125N06136,2000,5,7,12,13.6,131.4,55.0,980.0,658
563188,2000125N06136,2000,5,7,18,13.7,131.4,60.0,975.0,661
563190,2000125N06136,2000,5,8,0,14.0,131.6,65.0,970.0,691


# Bước 2. Tính toán tạo thành 87 feature (module 2. feature_engineering)

In [11]:
cols = ['SID', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'LAT', 'LON',
        'WMO_WIND', 'WMO_PRES', 'DIST2LAND']
df_fe = df[cols].copy()

df_fe = df_fe.sort_values(['SID', 'YEAR', 'MONTH', 'DAY', 'HOUR']).reset_index(drop=True)

In [12]:
for t in [6, 12, 18, 24]:
    step = t // 6
    df_fe[f'LAT_T-{t}']       = df_fe.groupby('SID')['LAT'].shift(step)
    df_fe[f'LON_T-{t}']       = df_fe.groupby('SID')['LON'].shift(step)
    df_fe[f'WIND_T-{t}']      = df_fe.groupby('SID')['WMO_WIND'].shift(step)
    df_fe[f'PRES_T-{t}']      = df_fe.groupby('SID')['WMO_PRES'].shift(step)

df_fe['DIST2LAND_T-6'] = df_fe.groupby('SID')['DIST2LAND'].shift(1)

In [13]:
for hours in [6, 12, 18, 24]:
    # Zonal and meridional displacement (degrees per interval)
    df_fe[f'V_Zonal_{hours}h']      = df_fe['LON'] - df_fe[f'LON_T-{hours}']
    df_fe[f'V_Meridional_{hours}h'] = df_fe['LAT'] - df_fe[f'LAT_T-{hours}']

    # Intensity tendency
    df_fe[f'Delta_PRES_{hours}h'] = df_fe['WMO_PRES'] - df_fe[f'PRES_T-{hours}']
    df_fe[f'Delta_WIND_{hours}h'] = df_fe['WMO_WIND'] - df_fe[f'WIND_T-{hours}']

    # Resultant translation speed magnitude
    df_fe[f'V_Actual_{hours}h'] = np.sqrt(
        df_fe[f'V_Zonal_{hours}h']**2 + df_fe[f'V_Meridional_{hours}h']**2
    )

In [14]:
for hours in [6, 12, 18, 24]:
    # Movement azimuth in degrees (-180 to +180, measured counter-clockwise from East)
    df_fe[f'Azimuth_{hours}h'] = np.degrees(
        np.arctan2(df_fe[f'V_Meridional_{hours}h'], df_fe[f'V_Zonal_{hours}h'])
    )

# Translational acceleration: change in the most-recent 6h velocity vector
# computed over 12h and 24h look-back windows.
df_fe['Acc_Zonal_12h']      = df_fe['V_Zonal_6h']      - df_fe.groupby('SID')['V_Zonal_6h'].shift(2)
df_fe['Acc_Meridional_12h'] = df_fe['V_Meridional_6h'] - df_fe.groupby('SID')['V_Meridional_6h'].shift(2)
df_fe['Acc_Zonal_24h']      = df_fe['V_Zonal_6h']      - df_fe.groupby('SID')['V_Zonal_6h'].shift(4)
df_fe['Acc_Meridional_24h'] = df_fe['V_Meridional_6h'] - df_fe.groupby('SID')['V_Meridional_6h'].shift(4)

# Resultant acceleration magnitude
df_fe['Acc_Actual_12h'] = np.sqrt(df_fe['Acc_Zonal_12h']**2 + df_fe['Acc_Meridional_12h']**2)
df_fe['Acc_Actual_24h'] = np.sqrt(df_fe['Acc_Zonal_24h']**2 + df_fe['Acc_Meridional_24h']**2)

In [15]:
def calc_angle_diff(angle1, angle2):
    """Signed angular difference in (-180, +180], handling 0/360 wrap-around."""
    return (angle1 - angle2 + 180) % 360 - 180

# Directional-change features: how much the storm has turned between windows.
# Positive = counter-clockwise (left turn); negative = clockwise (right turn / recurvature).
df_fe['Diff_Azimuth_6_12']  = calc_angle_diff(df_fe['Azimuth_6h'],  df_fe['Azimuth_12h'])
df_fe['Diff_Azimuth_12_24'] = calc_angle_diff(df_fe['Azimuth_12h'], df_fe['Azimuth_24h'])
df_fe['Diff_Azimuth_6_18']  = calc_angle_diff(df_fe['Azimuth_6h'],  df_fe['Azimuth_18h'])
df_fe['Diff_Azimuth_6_24']  = calc_angle_diff(df_fe['Azimuth_6h'],  df_fe['Azimuth_24h'])

In [16]:
# Zonal × meridional product features capture quadrant-specific behaviour
# (e.g. NW-moving storms vs NE-moving storms have different product signs).
df_fe['X55_Prod_V_6h']    = df_fe['V_Zonal_6h']    * df_fe['V_Meridional_6h']
df_fe['X56_Prod_V_12h']   = df_fe['V_Zonal_12h']   * df_fe['V_Meridional_12h']
df_fe['X57_Prod_V_18h']   = df_fe['V_Zonal_18h']   * df_fe['V_Meridional_18h']
df_fe['X58_Prod_V_24h']   = df_fe['V_Zonal_24h']   * df_fe['V_Meridional_24h']
df_fe['X59_Prod_Acc_12h'] = df_fe['Acc_Zonal_12h'] * df_fe['Acc_Meridional_12h']
df_fe['X60_Prod_Acc_24h'] = df_fe['Acc_Zonal_24h'] * df_fe['Acc_Meridional_24h']

In [17]:
# Temporary helper columns (dropped after use in the cleanup cell below).
_azimuth_past_12h = df_fe.groupby('SID')['Azimuth_6h'].shift(2)
_azimuth_past_24h = df_fe.groupby('SID')['Azimuth_6h'].shift(4)

# Angular acceleration — signed rate of direction change.
# Captures recurvature tendency: a large positive value means the storm is
# turning sharply counter-clockwise (poleward recurvature in the WP basin).
df_fe['X61_Angular_Acc_12h'] = calc_angle_diff(df_fe['Azimuth_6h'], _azimuth_past_12h)
df_fe['X62_Angular_Acc_24h'] = calc_angle_diff(df_fe['Azimuth_6h'], _azimuth_past_24h)

In [18]:
def haversine(lon1, lat1, lon2, lat2):
    """Great-circle distance (km) between (lon1, lat1) and (lon2, lat2)."""
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    return 6371.0 * 2.0 * np.arcsin(np.sqrt(a))

# Physical displacement (km) — more meaningful than raw degree differences
# because 1 degree of longitude covers less physical distance at higher latitudes.
df_fe['X63_Dist_6h']  = haversine(df_fe['LON'], df_fe['LAT'], df_fe['LON_T-6'],  df_fe['LAT_T-6'])
df_fe['X64_Dist_12h'] = haversine(df_fe['LON'], df_fe['LAT'], df_fe['LON_T-12'], df_fe['LAT_T-12'])
df_fe['X65_Dist_18h'] = haversine(df_fe['LON'], df_fe['LAT'], df_fe['LON_T-18'], df_fe['LAT_T-18'])
df_fe['X66_Dist_24h'] = haversine(df_fe['LON'], df_fe['LAT'], df_fe['LON_T-24'], df_fe['LAT_T-24'])

In [19]:
# Temporary helpers — not stored as columns.
_v_past_12h = df_fe.groupby('SID')['V_Actual_6h'].shift(2)
_v_past_24h = df_fe.groupby('SID')['V_Actual_6h'].shift(4)

# Speed acceleration: did the storm speed up or slow down?
df_fe['X67_Actual_Acc_12h'] = df_fe['V_Actual_6h'] - _v_past_12h
df_fe['X68_Actual_Acc_24h'] = df_fe['V_Actual_6h'] - _v_past_24h

In [20]:
# Distance components from a fixed reference point (Vietnam coast: 108.20 E, 16.05 N).
# Encodes basin-relative position, which correlates with large-scale steering-flow regimes.
REF_LON, REF_LAT = 108.20, 16.05
df_fe['Dist_Ref_Zonal']      = df_fe['LON'] - REF_LON
df_fe['Dist_Ref_Meridional'] = df_fe['LAT'] - REF_LAT
df_fe['Dist_Ref_Actual']     = np.sqrt(df_fe['Dist_Ref_Zonal']**2 + df_fe['Dist_Ref_Meridional']**2)

In [ ]:
df_fe['_Grid_LON'] = (df_fe['LON'] // 2) * 2
df_fe['_Grid_LAT'] = (df_fe['LAT'] // 2) * 2

years = sorted(df_fe['YEAR'].unique())
climatology_list = []

for current_year in years:
    historical_data   = df_fe[df_fe['YEAR'] < current_year]
    current_year_data = df_fe[df_fe['YEAR'] == current_year].copy()

    clim_cols = ['X79_Total_Storms', 'X81_Mean_V_Zonal',
                 'X82_Mean_V_Meridional', 'X84_Mean_Azimuth']

    if not historical_data.empty:
        stats = historical_data.groupby(['_Grid_LAT', '_Grid_LON']).agg(
            X79_Total_Storms      = ('SID',            'nunique'),
            X81_Mean_V_Zonal      = ('V_Zonal_6h',     'mean'),
            X82_Mean_V_Meridional = ('V_Meridional_6h','mean'),
            X84_Mean_Azimuth      = ('Azimuth_6h',     'mean'),
        ).reset_index()

        current_year_data = current_year_data.merge(
            stats, on=['_Grid_LAT', '_Grid_LON'], how='left'
        )
    else:
        # Year 2000 has no historical data; columns initialised to NaN, filled below.
        for col in clim_cols:
            current_year_data[col] = np.nan

    climatology_list.append(current_year_data)

df_fe = pd.concat(climatology_list, ignore_index=True)

clim_cols = ['X79_Total_Storms', 'X81_Mean_V_Zonal',
             'X82_Mean_V_Meridional', 'X84_Mean_Azimuth']
global_means = {}
for year in years:
    hist = df_fe[df_fe['YEAR'] < year]
    global_means[year] = hist[clim_cols].mean() if not hist.empty else pd.Series({c: 0.0 for c in clim_cols})

for year in years:
    mask = df_fe['YEAR'] == year
    df_fe.loc[mask, clim_cols] = df_fe.loc[mask, clim_cols].fillna(global_means[year])

cols_to_fix = df_fe.columns.difference(['SID'])
df_fe[cols_to_fix] = df_fe.groupby('SID')[cols_to_fix].transform(lambda x: x.bfill())
df_fe = df_fe.fillna(df_fe.mean(numeric_only=True))


df_fe = df_fe.drop(columns=['_Grid_LON', '_Grid_LAT'])

print("Total features (excl. SID + time + targets):", df_fe.shape[1])
print("NaN count after fillna:", df_fe.isnull().sum().sum())

Total features (excl. SID + time + targets): 82
NaN count after fillna: 0


In [23]:
df_fe.to_csv('data/clean/dynamic_feature.csv', index=False)
print("Saved -> data/clean/dynamic_feature.csv")
print(f"Shape: {df_fe.shape}")

Saved -> data/clean/dynamic_feature.csv
Shape: (10903, 82)
